## Step 0: Mounting Google Drive and Importing Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/multimodal-xray-agent
!ls

In [ ]:
!pip install -q vllm

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

In [4]:
# Access the secret token and login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face Hub.")
except userdata.SecretNotFoundError:
    print("Secret 'HF_TOKEN' not found. Please add it to the Colab Secrets manager.")
except Exception as e:
    print(f"An error occurred during authentication: {e}")

Successfully authenticated with Hugging Face Hub.


## Step 1: Loading the Quantized Model

In [5]:
MODEL_REPO_ID = "AMead10/Llama-3.2-3B-Instruct-AWQ"

In [ ]:
llm = LLM(
    model=MODEL_REPO_ID,
    dtype='auto',
    trust_remote_code=True,
    max_model_len=4096
)

## Step 1: Run an Inference Test

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO_ID)

In [8]:
# Simulate the context from your RAG pipeline
vision_caption_context = "The lungs appear hyperinflated with flattened hemidiaphragms. No focal consolidation or pleural effusion is seen."
similar_case_context = "Findings are suggestive of chronic obstructive pulmonary disease (COPD). The heart size is normal."
user_query_context = "Please provide a detailed impression of the chest X-ray."

In [9]:
user_prompt_content = f"""
You are an expert radiology assistant. Your task is to synthesize the following context into a single, concise, professional diagnostic impression.

### AI-Generated Image Caption:
{vision_caption_context}

### Impression from a Similar Case:
{similar_case_context}

### Original User Query:
{user_query_context}

Based on all the provided information, generate the final diagnostic impression:
"""

In [10]:
messages = [
    {"role": "user", "content": user_prompt_content}
]

In [11]:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
sampling_params = SamplingParams(max_tokens=396, temperature=0.2)
outputs = llm.generate(prompt, sampling_params)

In [13]:
# Print the Successful Output ---
print("\n--- vLLM Test Generation Output ---")
for output in outputs:
    generated_text = output.outputs[0].text
    print(generated_text.strip())


--- vLLM Test Generation Output ---
**Diagnostic Impression:**

The chest X-ray reveals hyperinflation of the lungs with flattened diaphragms, consistent with chronic obstructive pulmonary disease (COPD). The absence of focal consolidation or pleural effusion supports this diagnosis. The heart size appears normal, which is consistent with the expected findings in COPD. Overall, the radiographic appearance is consistent with a long-standing history of COPD.
